# EDM block analysis (PMT + CCD)

Analyses one cluster of EDM blocks:

1. loads each block zip (PMT) and the matching CCD csv files,
2. computes the gated signal and the asymmetry $(A-B)/(A+B)$ for every shot,
3. demodulates the asymmetry into switch channels ({E}, {B}, {DB}, {E·B}, ...),
4. converts the E-odd channels into an equivalent magnetic field and an eEDM estimate.

It only needs numpy, pandas and matplotlib (no pythonnet / SharedCode). Edit the settings cell and run all.

**The eEDM number is statistics-only**: no systematics, and it depends on the DB step calibration and $E_\mathrm{eff}$ set below.

In [ ]:
import zipfile, glob, os, itertools
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Settings

In [ ]:
cluster  = "13Nov2600"
blockDir = r"C:\Users\UEDM\Box\Ultracold eEDM\Data\BlockData\2026\November2026"
ccdDir   = r"C:\Users\UEDM\Box\Ultracold eEDM\Data\2026\2026-11\20261113\Blocks"

# PMT gates, in ms from the start of the TOF
signalGate     = (8, 45)
backgroundGate = (50, 114.59)   # same background gate as Block.AddDetectorsToBlock

# CCD frames (each shot has ccdNBurstFrames frames)
ccdSignalFrames     = range(0, 9)
ccdBackgroundFrames = range(10, 20)

# cuts
dropFirstShots = 8              # start-of-block transient
excludeBlocks  = list(range(7)) + [22]   # whole blocks to drop from the EDM estimate (0-6: very low signal, 22: probe oscillating)
excludeShots   = {27: range(18, 24)}   # {file index: shots} to drop

# conversion to an EDM
coilCalibration = None          # nT/mA for the DB step; None -> magnetCalibration from the block config (16.9).
                                # NB the Dec 2024 Mathematica analysis used 5 nT/mA - this choice scales the EDM directly.
dbStepPicoTesla = None          # set directly in pT to override the line above
Eeff = 18.0e9                   # V/cm, YbF at 20 kV/cm (14.5 GV/cm at 10 kV/cm)

## Loading blocks

In [ ]:
def readBlock(path):
    # Read a BlockHead zip into numpy arrays without needing SharedCode.
    tofA, tofB, spd, cfg, mods = [], [], [], {}, {}
    with zipfile.ZipFile(path) as z, z.open("block.xml") as f:
        for _, el in ET.iterparse(f):
            if el.tag == "EDMPoint":
                tofs = el.find("Shot").find("TOFs")
                tofA.append([float(s.text) for s in tofs[0].find("Data")])
                tofB.append([float(s.text) for s in tofs[1].find("Data")])
                spd.append({it.findtext("key"): it.findtext("value") for it in el.find("SinglePointData").iter("item")})
                if len(tofA) == 1:
                    gateStart = int(tofs[0].findtext("GateStartTime"))
                    clockPeriod = int(tofs[0].findtext("ClockPeriod"))
                el.clear()
            elif el.tag == "Config":
                cfg = {it.findtext("key"): it.findtext("value") for it in el.find("Settings").iter("item")}
                for m in list(el.find("DigitalModulations")) + list(el.find("AnalogModulations")):
                    w = m.find("Waveform")
                    mods[m.findtext("Name")] = dict(code=[b.text == "true" for b in w.find("Code")],
                                                    inverted=w.findtext("Inverted") == "true",
                                                    step=m.findtext("Step"))
    A, B = np.array(tofA), np.array(tofB)
    t = (gateStart + np.arange(A.shape[1]) * clockPeriod) / 1000   # ms, same as SharedCode TOF.Times
    return dict(A=A, B=B, t=t, cfg=cfg, mods=mods, spd=pd.DataFrame(spd))


def waveformBits(code, inverted):
    # Same as SharedCode Waveform.GenerateBits, returned as +1/-1 per point.
    n = len(code); bits = np.ones(2**n, bool)
    for i, c in enumerate(code):
        if c:
            inc = 2**(n - 1 - i)
            for k in range(0, 2**n, 2 * inc):
                bits[k + inc:k + 2 * inc] = ~bits[k + inc:k + 2 * inc]
    if inverted:
        bits = ~bits
    return np.where(bits, 1, -1)


def readCCD(index, camera, nFrames):
    path = os.path.join(ccdDir, f"{cluster}_{index:05d}_CCD{camera}.csv")
    if not os.path.exists(path):
        return None
    counts = pd.read_csv(path).iloc[:, 2].to_numpy(float)
    return counts.reshape(-1, nFrames)       # (shots, frames)

In [ ]:
files = sorted(glob.glob(os.path.join(blockDir, f"{cluster}_*.zip")))
blocks = {}
for path in files:
    index = int(os.path.basename(path).split("_")[1])
    try:
        b = readBlock(path)
    except Exception as e:
        print(f"skipping {os.path.basename(path)}: {e}")
        continue
    nFrames = int(b["cfg"].get("ccdNBurstFrames", 20))
    b["ccdA"], b["ccdB"] = readCCD(index, "A", nFrames), readCCD(index, "B", nFrames)
    blocks[index] = b
    print(f"block {index}: {len(b['A'])} shots, CCD {'yes' if b['ccdA'] is not None else 'no'}")

## Per-shot signals

In [ ]:
for i, b in blocks.items():
    t = b["t"]; dt = t[1] - t[0]
    sig = (t >= signalGate[0]) & (t < signalGate[1])
    bg  = (t >= backgroundGate[0]) & (t < backgroundGate[1])
    for d in "AB":
        background = b[d][:, bg].mean(axis=1)
        b["bg" + d] = background
        b["s" + d] = (b[d][:, sig] - background[:, None]).sum(axis=1) * dt     # V ms
    b["asym"] = (b["sA"] - b["sB"]) / (b["sA"] + b["sB"])

    if b["ccdA"] is not None and len(b["ccdA"]) == len(b["A"]):
        f0, f1 = list(ccdSignalFrames), list(ccdBackgroundFrames)
        for d in "AB":
            c = b["ccd" + d]
            b["q" + d] = c[:, f0].sum(axis=1) - len(f0) * c[:, f1].mean(axis=1)
        b["ccdAsym"] = (b["qA"] - b["qB"]) / (b["qA"] + b["qB"])
    else:
        b["ccdAsym"] = None

In [ ]:
summary = pd.DataFrame([dict(block=i, sA=b["sA"].mean(), sB=b["sB"].mean(), bgA=b["bgA"].mean(), bgB=b["bgB"].mean(),
                             asym=b["asym"].mean()) for i, b in blocks.items()]).set_index("block")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(summary.index, summary.sA, "o-", label="PMT A")
ax[0].plot(summary.index, summary.sB, "o-", label="PMT B")
ax[0].set_yscale("log"); ax[0].set_xlabel("block"); ax[0].set_ylabel("signal per shot (V ms)"); ax[0].legend()
x0 = 0
for i, b in blocks.items():
    x = x0 + np.arange(len(b["sA"])); x0 += len(b["sA"])
    ax[1].plot(x, b["sA"], c="C0", lw=.5); ax[1].plot(x, b["sB"], c="C1", lw=.5)
    ax[1].text(x.mean(), 0, str(i), ha="center", va="bottom", fontsize=7)
ax[1].set_xlabel("shot (all blocks)"); ax[1].set_ylabel("V ms")
plt.tight_layout(); plt.show()
summary.round(4)

Look at single blocks here, e.g. to find shots to add to `excludeShots`.

In [ ]:
i = 22
b = blocks[i]
fig, ax = plt.subplots(2, 1, figsize=(13, 5), sharex=True)
ax[0].plot(b["sA"], label="PMT A"); ax[0].plot(b["sB"], label="PMT B"); ax[0].set_ylabel("V ms"); ax[0].legend()
ax[1].plot(b["asym"], label="PMT")
if b["ccdAsym"] is not None:
    ax[1].plot(b["ccdAsym"], label="CCD")
ax[1].set_ylabel("asymmetry"); ax[1].set_xlabel("shot"); ax[1].legend(); ax[0].set_title(f"block {i}")
plt.show()

## Switch-channel demodulation

Each shot's asymmetry is fitted to all 16 products of the E, B, DB and StirapAOM waveforms. The first `dropFirstShots` shots and any `excludeShots` are left out of the fit.

In [ ]:
switches = ["E", "B", "DB", "StirapAOM"]

def demodulate(x, waveforms, keep):
    names, cols = [], []
    for r in range(len(switches) + 1):
        for combo in itertools.combinations(switches, r):
            names.append(".".join(combo) if combo else "SIG")
            cols.append(np.prod([waveforms[s] for s in combo], axis=0) if combo else np.ones(len(x)))
    X = np.stack(cols, axis=1)[keep]
    beta, *_ = np.linalg.lstsq(X, x[keep], rcond=None)
    resid = x[keep] - X @ beta
    noise = resid.std(ddof=X.shape[1])
    err = np.sqrt(np.diag(np.linalg.inv(X.T @ X))) * noise
    return pd.Series(beta, names), pd.Series(err, names), noise

rows = []
for i, b in blocks.items():
    waveforms = {s: waveformBits(b["mods"][s]["code"], b["mods"][s]["inverted"]) for s in switches}
    keep = np.ones(len(b["asym"]), bool)
    keep[:dropFirstShots] = False
    keep[list(excludeShots.get(i, []))] = False
    for det, x in [("PMT", b["asym"]), ("CCD", b["ccdAsym"])]:
        if x is None:
            continue
        val, err, noise = demodulate(x, waveforms, keep)
        rows.append(dict(block=i, detector=det, noise=noise, **val, **err.add_suffix("_err")))
channels = pd.DataFrame(rows)
channels[["block", "detector", "noise", "SIG", "B", "DB", "B.DB", "E", "E.B", "E.DB"]].round(4)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for det, mk in [("PMT", "o"), ("CCD", "s")]:
    c = channels[channels.detector == det]
    ax[0].plot(c.block, c.noise, mk + "-", label=det)
    ax[1].errorbar(c.block, c["DB"], c["DB_err"], fmt=mk, label=det + " {DB}")
    ax[1].errorbar(c.block, c["B.DB"], c["B.DB_err"], fmt=mk, mfc="none", label=det + " {B.DB}")
    ax[2].errorbar(c.block, c["E.B"], c["E.B_err"], fmt=mk, label=det)
ax[0].set_yscale("log"); ax[0].set_title("asymmetry noise per shot")
ax[1].set_title("fringe slope channels"); ax[2].set_title("{E.B}")
for a in ax:
    a.set_xlabel("block"); a.legend(fontsize=8)
ax[1].axhline(0, c="k", lw=.5); ax[2].axhline(0, c="k", lw=.5)
plt.tight_layout(); plt.show()

## eEDM estimate

The DB step reverses with the B state (it changes |B|), so an EDM appears in {E·B}, calibrated by {DB}. Using both B states:

$$\kappa = \frac{\{E\cdot B\}\{DB\} + \{E\}\{B\cdot DB\}}{\{DB\}^2 + \{B\cdot DB\}^2}, \qquad \delta B_E = \kappa\,\Delta B_{DB}, \qquad d_e = \frac{\mu_B\,\delta B_E}{E_\mathrm{eff}}$$

Blocks are combined with a weighted mean; the error is scaled up by $\sqrt{\chi^2_\nu}$ if that is above 1.

In [ ]:
first = blocks[min(blocks)]
if dbStepPicoTesla is None:
    cal = coilCalibration if coilCalibration is not None else float(first["cfg"]["magnetCalibration"])
    dbStepPicoTesla = float(first["mods"]["DB"]["step"]) * cal * 1000
muB, e = 9.274e-24, 1.602e-19                       # J/T, C
eCmPerPicoTesla = muB * 1e-12 / (e * Eeff)          # e cm per pT
print(f"DB step = {dbStepPicoTesla:.1f} pT,  1 pT = {eCmPerPicoTesla:.2e} e cm")

results = []
for det in ["PMT", "CCD"]:
    c = channels[(channels.detector == det) & ~channels.block.isin(excludeBlocks)]
    if c.empty:
        continue
    den = c["DB"]**2 + c["B.DB"]**2
    kappa = (c["E.B"] * c["DB"] + c["E"] * c["B.DB"]) / den
    kappaErr = c["E.B_err"] / np.sqrt(den)
    w = 1 / kappaErr**2
    mean = (kappa * w).sum() / w.sum()
    err = 1 / np.sqrt(w.sum())
    chi2 = ((kappa - mean)**2 * w).sum() / (len(c) - 1)
    err *= max(1, np.sqrt(chi2))
    dB, dBerr = mean * dbStepPicoTesla, err * dbStepPicoTesla
    results.append(dict(detector=det, blocks=len(c), chi2r=chi2,
                        dB_pT=dB, dB_err_pT=dBerr,
                        de=dB * eCmPerPicoTesla, de_err=dBerr * eCmPerPicoTesla,
                        de_90CL=(abs(dB) + 1.645 * dBerr) * eCmPerPicoTesla))
results = pd.DataFrame(results).set_index("detector")
results

PMT and CCD see the same molecules, so their results are not independent; don't average them.